# Waste Sorting with a Pretrained CNN

Transfer-learning pipeline for the [Kaggle Garbage Classification (12 classes)](https://www.kaggle.com/datasets/mostafaabla/garbage-classification) dataset.

**Backbone:** EfficientNetB0 (swap out for MobileNetV2 / ResNet50 in the config cell if you prefer — the preprocessing function is picked automatically to match).

**Pipeline:**
1. Load data from per-class folders, inspect class balance
2. Train / validation / test split (stratified, done once, saved to disk as file lists so it's reproducible)
3. Phase 1: freeze backbone, train new classification head
4. Phase 2: unfreeze last N backbone layers, fine-tune at low LR
5. Evaluate with per-class precision/recall/F1 + full confusion matrix
6. Save the trained model for the Streamlit app

> Run this in Colab/Kaggle with a GPU runtime. Update `DATA_DIR` to point at the extracted dataset.

## 1. Setup

In [ ]:
!pip -q install split-folders

import os, json, random, pathlib
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TF version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

In [ ]:
# ---- Config ----
DATA_DIR = "garbage_classification"   # folder containing one subfolder per class
BACKBONE = "efficientnet"             # "efficientnet" | "mobilenet" | "resnet50"
IMG_SIZE = {"efficientnet": (224, 224), "mobilenet": (224, 224), "resnet50": (224, 224)}[BACKBONE]
BATCH_SIZE = 32
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15   # remaining ~0.70 goes to train
HEAD_EPOCHS = 8
FINETUNE_EPOCHS = 10
UNFREEZE_LAST_N = 30   # number of backbone layers to unfreeze in phase 2

CLASS_NAMES = sorted(os.listdir(DATA_DIR))
print(f"{len(CLASS_NAMES)} classes:", CLASS_NAMES)

## 2. Explore class counts and sample images

Do this before any preprocessing. The classes are not evenly sized (e.g. `trash` and `battery`
tend to be far smaller than `paper`/`cardboard`), so this shapes both the split and the
class-weighting decision later.

In [ ]:
counts = {c: len(os.listdir(os.path.join(DATA_DIR, c))) for c in CLASS_NAMES}
for c, n in sorted(counts.items(), key=lambda x: -x[1]):
    print(f"{c:12s} {n}")

plt.figure(figsize=(10, 4))
plt.bar(counts.keys(), counts.values())
plt.xticks(rotation=45, ha='right')
plt.ylabel("images")
plt.title("Class distribution")
plt.tight_layout()
plt.show()

In [ ]:
# A handful of sample images per class
fig, axes = plt.subplots(len(CLASS_NAMES), 4, figsize=(10, 2.2 * len(CLASS_NAMES)))
for i, c in enumerate(CLASS_NAMES):
    files = os.listdir(os.path.join(DATA_DIR, c))[:4]
    for j, f in enumerate(files):
        img = keras.utils.load_img(os.path.join(DATA_DIR, c, f))
        axes[i, j].imshow(img)
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_ylabel(c)
    axes[i, 0].set_title(c, loc='left', fontsize=9)
plt.tight_layout()
plt.show()

## 3. Train / validation / test split

Splitting on the folder structure directly, stratified per class, so every class is
represented proportionally in all three sets. `split-folders` copies files into
`data_split/{train,val,test}/<class>/...` which we then load with
`image_dataset_from_directory`.

In [ ]:
import splitfolders

splitfolders.ratio(
    DATA_DIR,
    output="data_split",
    seed=SEED,
    ratio=(1 - VAL_SPLIT - TEST_SPLIT, VAL_SPLIT, TEST_SPLIT),
    group_prefix=None,
)

for split in ["train", "val", "test"]:
    total = sum(len(os.listdir(os.path.join("data_split", split, c))) for c in CLASS_NAMES)
    print(split, total)

In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    "data_split/train", labels="inferred", label_mode="categorical",
    class_names=CLASS_NAMES, image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED, shuffle=True,
)
val_ds = keras.utils.image_dataset_from_directory(
    "data_split/val", labels="inferred", label_mode="categorical",
    class_names=CLASS_NAMES, image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED, shuffle=False,
)
test_ds = keras.utils.image_dataset_from_directory(
    "data_split/test", labels="inferred", label_mode="categorical",
    class_names=CLASS_NAMES, image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED, shuffle=False,
)

## 4. Preprocessing matched to the backbone

Each pretrained backbone expects a specific input range / normalization. Using the wrong
one (e.g. plain `/255.`) silently degrades transfer learning, since the backbone's learned
filters were tuned for its own preprocessing.

In [ ]:
BACKBONES = {
    "efficientnet": (keras.applications.EfficientNetB0, keras.applications.efficientnet.preprocess_input),
    "mobilenet":    (keras.applications.MobileNetV2,   keras.applications.mobilenet_v2.preprocess_input),
    "resnet50":     (keras.applications.ResNet50,      keras.applications.resnet50.preprocess_input),
}
BackboneClass, preprocess_input = BACKBONES[BACKBONE]

def preprocess(x, y):
    return preprocess_input(x), y

AUTOTUNE = tf.data.AUTOTUNE

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

def prep_train(x, y):
    x = data_augmentation(x)
    return preprocess_input(x), y

train_ds = train_ds.map(prep_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds   = val_ds.map(preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds  = test_ds.map(preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

## 5. Class weights (handling imbalance at training time)

In addition to reporting per-class metrics later, we counteract imbalance during training
itself by weighting the loss so rare classes (e.g. `battery`, `trash`) aren't drowned out
by common ones (e.g. `paper`, `cardboard`).

In [ ]:
labels_flat = []
for c_idx, c in enumerate(CLASS_NAMES):
    labels_flat += [c_idx] * len(os.listdir(os.path.join("data_split/train", c)))
labels_flat = np.array(labels_flat)

weights = compute_class_weight(class_weight="balanced", classes=np.arange(len(CLASS_NAMES)), y=labels_flat)
class_weight = {i: w for i, w in enumerate(weights)}
class_weight

## 6. Build the model — Phase 1: frozen backbone + new head

In [ ]:
base_model = BackboneClass(
    include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,)
)
base_model.trainable = False   # freeze entirely for phase 1

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(len(CLASS_NAMES), activation="softmax")(x)
model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
]

history_head = model.fit(
    train_ds, validation_data=val_ds,
    epochs=HEAD_EPOCHS, class_weight=class_weight, callbacks=callbacks,
)

## 7. Phase 2 — unfreeze and fine-tune

Unfreezing the *whole* backbone tends to overfit fast on ~15k images and risks destroying
the ImageNet features; unfreezing *nothing further* leaves the backbone's higher-level
features un-adapted to waste photos specifically. Unfreezing the last `UNFREEZE_LAST_N`
layers is a middle ground: those layers hold the more task-specific, higher-level features,
while the earliest layers (generic edges/textures/colors) stay frozen. Combined with a much
lower learning rate, this nudges the backbone toward waste imagery without wrecking it.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-UNFREEZE_LAST_N]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),   # much lower LR for fine-tuning
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history_finetune = model.fit(
    train_ds, validation_data=val_ds,
    epochs=FINETUNE_EPOCHS, class_weight=class_weight, callbacks=callbacks,
)

In [ ]:
def plot_history(h1, h2, key):
    vals = h1.history[key] + h2.history[key]
    val_vals = h1.history[f"val_{key}"] + h2.history[f"val_{key}"]
    plt.plot(vals, label=f"train_{key}")
    plt.plot(val_vals, label=f"val_{key}")
    plt.axvline(len(h1.history[key]) - 0.5, color="gray", linestyle="--", label="fine-tune starts")
    plt.legend(); plt.title(key); plt.xlabel("epoch")

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1); plot_history(history_head, history_finetune, "accuracy")
plt.subplot(1, 2, 2); plot_history(history_head, history_finetune, "loss")
plt.tight_layout(); plt.show()

## 8. Evaluation on the held-out test set

Test set was never touched during training or validation. Accuracy alone hides how the
model handles the smaller classes, so we look at per-class precision/recall/F1 and the
full 12x12 confusion matrix.

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f} | Test loss: {test_loss:.4f}")

In [ ]:
y_true, y_pred = [], []
for x_batch, y_batch in test_ds:
    preds = model.predict(x_batch, verbose=0)
    y_true += list(np.argmax(y_batch.numpy(), axis=1))
    y_pred += list(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix — test set")
plt.xticks(rotation=45, ha="right"); plt.tight_layout(); plt.show()

### Reading the confusion matrix

Fill this in from your actual run — the off-diagonal entries worth writing up are almost
always: the three glass colors confused with each other and with clear plastic, and
`paper` vs `cardboard`. Both are genuinely a texture/color-based ambiguity at the pixel
level rather than a modeling bug: brown glass vs. green glass differs mainly in a subtle
hue shift a CNN can miss under different lighting, and crumpled paper vs. thin cardboard
share edges, fold lines and off-white color. Note the specific rows/columns your matrix
lit up, and whether the biggest confusions are exactly the ones this brief predicted, or
something else your data surfaced.

## 9. Save the trained model (used by the Streamlit app)

In [ ]:
model.save("waste_classifier.keras")

with open("class_names.json", "w") as f:
    json.dump({"backbone": BACKBONE, "img_size": IMG_SIZE, "class_names": CLASS_NAMES}, f, indent=2)

print("Saved waste_classifier.keras and class_names.json")